# Transform Drivers Data

1. Read bronze `drivers` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfbirth` → `date_of_birth`)
4. Concatenate `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform the value to Title Case
5. Remove duplicate records
6. Transform values of column `nationality` to Title Case
7. Write the transformed data to silver `drivers` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read bronze `drivers` table

In [0]:
drivers_df = spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
drivers_dropped_df = drivers_df.drop("url")

#### Step 3 - Standardise Column Names
- Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfBirth` → `date_of_birth`)

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"
        })
)

#### Step 4 - Concatenate name.givenName and name.familyName to create a new column called driver_name

In [0]:
drivers_concat_df = (
    drivers_renamed_df
        .withColumn("driver_name", F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
        .drop("name")
)

#### Step 5 - Remove duplicate records

In [0]:
drivers_distinct_df = drivers_concat_df.dropDuplicates(["driver_id"])
display(drivers_distinct_df)

#### Step 7 - Transform values of columns `nationality` to Title Case

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn('nationality', F.initcap(F.col('nationality')))
)
display(drivers_final_df)

#### Step 8 - Write the transformed data to silver `drivers` table

In [0]:
(
    drivers_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))